# La señal de audio y su transcripción

**Lección 1 · Clase 5.3** — antes de que un modelo entienda una palabra, la voz es *una onda de presión convertida en números*. Esta lección parte ahí: mirar la señal, entender de qué está hecha, y solo entonces pasarla por un modelo de transcripción.

¿Por qué no ir directo a la API? Porque casi todo lo que sale mal en un proyecto de voz sale mal **en la señal**, no en el modelo: audio mal muestreado, un micrófono saturado, un canal telefónico que corta los agudos, ruido de fondo de una sucursal. Si sabes cómo se ve la señal, sabes dónde mirar cuando la transcripción sale mala.

El plan:

| | |
|---|---|
| **La señal** | Muestreo, waveform y espectrograma de un fragmento musical. |
| **La voz** | El mismo análisis sobre voz: qué la hace distinta de la música. |
| **El texto** | `gpt-transcribe` convierte la señal en texto. |
| **El límite** | Degradamos la señal con ruido hasta que el modelo se equivoca. |

In [ ]:
# Esta lección usa el entorno uv del README. Si la corres en Colab, descomenta:
# %pip install -q openai==2.52.0 numpy==2.5.1 scipy==1.18.0 matplotlib==3.11.1 python-dotenv==1.2.2
from dotenv import load_dotenv
import os

# Carga OPENAI_API_KEY desde .env si existe (local); en Colab usa Secrets.
load_dotenv()

try:
    from google.colab import userdata  # type: ignore
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY", "")
    except Exception:
        pass
except Exception:
    pass

HAY_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
print("OPENAI_API_KEY presente:", HAY_OPENAI)
if not HAY_OPENAI:
    print("⚠️ Sin llave el análisis de la señal corre igual (es 100% local);")
    print("   solo se saltan las celdas de transcripción.")

## Qué es, exactamente, una señal de audio

El micrófono mide la presión del aire miles de veces por segundo. Cada medición se guarda como un número entero. Eso es todo: **un audio es un array de números**.

Tres parámetros lo definen y explican casi todos los problemas prácticos:

- **Frecuencia de muestreo** (*sample rate*): cuántas mediciones por segundo. 44.100 Hz en un CD, 24.000 Hz en lo que devuelve la API de OpenAI, **8.000 Hz** en telefonía tradicional. El teorema de Nyquist pone el techo: solo puedes representar frecuencias de hasta la mitad del sample rate. Por eso una llamada telefónica suena opaca — todo lo que está sobre 4 kHz simplemente no existe en la grabación.
- **Profundidad de bits** (*bit depth*): cuánto rango tiene cada número. PCM de 16 bits → enteros entre −32.768 y 32.767. Si la señal se pasa de ese rango, *clipea*: se corta plana y la información se pierde para siempre.
- **Canales**: mono o estéreo. Para voz casi siempre da igual, y por eso los modelos de transcripción mezclan a mono.

Empecemos con un fragmento musical, que tiene un espectro más rico y hace visible lo que después buscaremos en la voz.

In [ ]:
from pathlib import Path
import urllib.request

import numpy as np
from scipy.io import wavfile
from IPython.display import Audio, display

BASE_RAW = (
    "https://raw.githubusercontent.com/josepenam/clases-diplomado-gen-ia/main/"
    "class_5_3_voz/leccion1_audio_y_transcripcion/data/"
)


def asegurar(nombre: str) -> Path:
    """Devuelve la ruta local del insumo; en Colab lo baja del repo público."""
    ruta = Path("data") / nombre
    if not ruta.exists():
        ruta.parent.mkdir(parents=True, exist_ok=True)
        pedido = urllib.request.Request(
            BASE_RAW + nombre, headers={"User-Agent": "Mozilla/5.0 (clase-diplomado-gen-ia)"}
        )
        ruta.write_bytes(urllib.request.urlopen(pedido).read())
    return ruta


ruta_musica = asegurar("muestra_musical.wav")
sample_rate, musica = wavfile.read(ruta_musica)

if musica.ndim == 2:  # si viniera en estéreo, nos quedamos con un canal
    musica = musica[:, 0]

print(f"Frecuencia de muestreo : {sample_rate:,} Hz")
print(f"Muestras               : {len(musica):,}")
print(f"Duración               : {len(musica) / sample_rate:.2f} s")
print(f"Tipo de dato           : {musica.dtype}  (rango {musica.min()} a {musica.max()})")
print(f"Frecuencia máxima representable (Nyquist): {sample_rate // 2:,} Hz")
display(Audio(filename=str(ruta_musica)))

### El waveform: amplitud en el tiempo

La vista más directa de esos números. Dice **cuándo** hay energía y cuánta — se ven los ataques, los silencios, si la grabación viene muy baja o clipeada — pero no dice **qué** suena: un piano y una voz pueden tener waveforms parecidos.

In [ ]:
import matplotlib.pyplot as plt

tiempo = np.linspace(0, len(musica) / sample_rate, num=len(musica))

plt.figure(figsize=(12, 3.5))
plt.plot(tiempo, musica, linewidth=0.5)
plt.title("Waveform — fragmento musical")
plt.xlabel("Tiempo [s]")
plt.ylabel("Amplitud (PCM 16 bits)")
plt.margins(x=0)
plt.tight_layout()
plt.show()

### El espectrograma: qué frecuencias, y cuándo

Acá está la idea que sostiene todo el reconocimiento de voz. En vez de mirar la señal completa, se corta en ventanas de unas decenas de milisegundos y a cada ventana se le aplica una **transformada de Fourier**: qué frecuencias la componen y con cuánta energía. Apilando esas ventanas se obtiene una imagen — tiempo en el eje X, frecuencia en el Y, energía en el color.

Ese cambio de representación es el punto clave: **el espectrograma es una imagen**, y por eso los modelos de audio pudieron reutilizar directamente la maquinaria de la visión por computador que vimos en la clase 5.2. Whisper, por ejemplo, no consume audio crudo: consume un espectrograma de mel.

In [ ]:
from scipy.signal import spectrogram


def dibujar_espectrograma(senal, sr, titulo, techo_hz=None):
    frecuencias, tiempos, Sxx = spectrogram(senal.astype(float), fs=sr, nperseg=1024)
    plt.figure(figsize=(11, 4))
    # 1e-10 evita log10(0) en los silencios
    plt.pcolormesh(tiempos, frecuencias, 10 * np.log10(Sxx + 1e-10), shading="gouraud")
    plt.colorbar(label="Energía [dB]")
    plt.title(titulo)
    plt.xlabel("Tiempo [s]")
    plt.ylabel("Frecuencia [Hz]")
    plt.ylim(0, techo_hz or sr // 2)
    plt.tight_layout()
    plt.show()


dibujar_espectrograma(musica, sample_rate, "Espectrograma — fragmento musical")

## La voz se ve distinta

Ahora el mismo análisis sobre voz. Hay tres cosas que buscar, y las tres explican por qué la voz es un tipo de señal particular:

- **Franjas horizontales apiladas**: son los armónicos. La más baja es la **frecuencia fundamental** (F0), el tono de la persona — típicamente 85-180 Hz en voces masculinas y 165-255 Hz en femeninas.
- **Manchas de energía concentrada**: son los **formantes**, las resonancias que la boca y la lengua imprimen sobre esos armónicos. Los formantes son lo que distingue una "a" de una "i". Un modelo de transcripción, en el fondo, lee formantes.
- **Casi toda la energía útil bajo los 4 kHz**: por eso la telefonía a 8 kHz funciona — feo, pero inteligible.

Fíjate también en que el espectrograma de voz es mucho más *vacío* que el musical: la voz es una señal escasa, con silencios entre sílabas.

In [ ]:
ruta_voz = asegurar("muestra_voz.wav")
sr_voz, voz = wavfile.read(ruta_voz)
if voz.ndim == 2:
    voz = voz[:, 0]

print(f"{sr_voz:,} Hz · {len(voz) / sr_voz:.2f} s · Nyquist {sr_voz // 2:,} Hz")
display(Audio(filename=str(ruta_voz)))

tiempo_voz = np.linspace(0, len(voz) / sr_voz, num=len(voz))
plt.figure(figsize=(12, 3))
plt.plot(tiempo_voz, voz, linewidth=0.5, color="tab:orange")
plt.title("Waveform — voz (se ven las sílabas y los silencios entre ellas)")
plt.xlabel("Tiempo [s]")
plt.ylabel("Amplitud")
plt.margins(x=0)
plt.tight_layout()
plt.show()

# Techo de 5 kHz: ahí vive casi toda la información fonética
dibujar_espectrograma(voz, sr_voz, "Espectrograma — voz (armónicos y formantes)", techo_hz=5000)

## De la señal al texto

Durante décadas esto se hizo con modelos acústicos, diccionarios de fonemas y modelos de lenguaje separados, cada uno afinado a mano. **Whisper** (OpenAI, 2022) lo colapsó en un solo transformer entrenado sobre 680.000 horas de audio multilingüe, y de ahí en adelante transcribir dejó de ser un problema de ingeniería para volverse una llamada de API.

Hoy la familia está especializada por caso de uso:

| Modelo | Cuándo usarlo |
|---|---|
| **`gpt-transcribe`** | Transcripción de **archivos**. Es el punto de partida recomendado, y el que usamos acá. |
| `gpt-realtime-whisper` | **Streaming**: entrega deltas de texto mientras la persona habla, con latencia sub-segundo. Para subtítulos en vivo. |
| `gpt-4o-transcribe-diarize` | Cuando necesitas saber **quién** habló en cada tramo (`diarized_json`). |
| `whisper-1` | El original; sigue disponible y es el único con endpoint de **traducción** directa. |

La llamada es de tres líneas: abrir el archivo y pedirla.

In [ ]:
from openai import OpenAI

MODELO_STT = "gpt-transcribe"
transcripcion = None

if not HAY_OPENAI:
    print("⚠️ Falta OPENAI_API_KEY — se salta la transcripción (y las celdas que dependen de ella).")
else:
    cliente = OpenAI()  # lee OPENAI_API_KEY del entorno
    with open(ruta_voz, "rb") as archivo:
        respuesta = cliente.audio.transcriptions.create(model=MODELO_STT, file=archivo)
    transcripcion = respuesta.text
    print(f"[{MODELO_STT}] {transcripcion}")

## El límite: la señal es el techo del modelo

Nuestra muestra es **modo fácil** y hay que decirlo: es voz sintética, grabada sin ruido, sin acento marcado, sin nadie hablando encima. Ningún audio real de una empresa se parece a esto — una llamada de call center trae ruido de sala, compresión telefónica, gente interrumpiéndose y nombres propios que ningún modelo de lenguaje predice.

Volvamos entonces a la señal, ahora para romperla a propósito. Le sumamos ruido blanco a distintas **relaciones señal/ruido** (SNR) y vemos a partir de qué punto la transcripción se cae — y, más importante, **de qué forma** se cae. Es el experimento que une las dos mitades de la lección: el modelo no puede recuperar información que la señal ya no contiene, pero eso no significa que se quede callado.

$$\mathrm{SNR_{dB}} = 10 \cdot \log_{10}\!\left(\frac{P_{\text{señal}}}{P_{\text{ruido}}}\right)$$

Como referencia: una oficina tranquila ronda los 30 dB de SNR, una calle o un café los 10 dB, y bajo 0 dB el ruido ya es **más fuerte que la voz**. Vale la pena bajar hasta niveles absurdos para ver dónde está de verdad el límite.

Un detalle del diseño del experimento: transcribimos **cada archivo tres veces**. El ruido tiene semilla fija, así que el audio es idéntico byte por byte en las tres llamadas — lo único que cambia es el modelo, que no es determinista. Si las tres respuestas coinciden, el modelo está seguro; si divergen, estamos en la zona donde ya está adivinando.

In [ ]:
import scipy.io.wavfile as wavwrite

SALIDAS = Path("outputs")
SALIDAS.mkdir(exist_ok=True)

generador = np.random.default_rng(42)  # semilla fija: el experimento es reproducible
voz_float = voz.astype(float)
potencia_senal = np.mean(voz_float**2)


def con_ruido(snr_db: float) -> np.ndarray:
    """Devuelve la voz contaminada con ruido blanco a la SNR pedida."""
    potencia_ruido = potencia_senal / (10 ** (snr_db / 10))
    ruido = generador.normal(0, np.sqrt(potencia_ruido), size=voz_float.shape)
    sucia = voz_float + ruido
    # Normalizamos solo si nos pasamos del rango PCM16, para no clipear
    pico = np.max(np.abs(sucia))
    if pico > 32767:
        sucia = sucia * (32767 / pico)
    return sucia.astype(np.int16)


NIVELES_SNR = [20, 10, 5, 0, -5, -10, -15]
muestras_sucias = {}

for snr in NIVELES_SNR:
    sucia = con_ruido(snr)
    destino = SALIDAS / f"voz_snr_{snr}dB.wav"
    wavwrite.write(destino, sr_voz, sucia)
    muestras_sucias[snr] = destino
    print(f"SNR {snr:>3} dB → {destino}")

# Cómo se ve el peor caso comparado con el original
PEOR = NIVELES_SNR[-1]

dibujar_espectrograma(voz, sr_voz, "Original", techo_hz=5000)
dibujar_espectrograma(
    wavfile.read(muestras_sucias[PEOR])[1], sr_voz,
    f"SNR {PEOR} dB — el ruido blanco tapa el espectro completo", techo_hz=5000,
)

In [ ]:
REPETICIONES = 3  # el mismo archivo, tres llamadas: el modelo no es determinista

if not HAY_OPENAI:
    print("⚠️ Falta OPENAI_API_KEY — se salta el experimento de ruido.")
else:
    print(f"Referencia (sin ruido): {transcripcion}\n")
    for snr, ruta in muestras_sucias.items():
        print(f"SNR {snr:>4} dB")
        for intento in range(1, REPETICIONES + 1):
            with open(ruta, "rb") as archivo:
                texto = cliente.audio.transcriptions.create(model=MODELO_STT, file=archivo).text
            print(f"   {intento}. {texto.strip() or '(vacío)'}")
        print()

    print("Escucha el peor caso para calibrar tu propio oído contra el del modelo:")
    display(Audio(filename=str(muestras_sucias[PEOR])))

### Cómo leer ese resultado

La degradación tiene **tres etapas**, y la del medio es la que hay que llevarse a la casa.

**1. Robustez sorprendente (20 dB → 0 dB).** El modelo transcribe perfecto incluso a 0 dB, donde el ruido tiene tanta potencia que la voz apenas se distingue al oído humano. Las tres repeticiones dan el mismo texto: el modelo está seguro. Esta robustez es consecuencia directa de las 680.000 horas de entrenamiento — buena parte de ese corpus era audio malo, así que aprendió a oír en el ruido mejor que muchas personas. El tramo cubre cualquier oficina, café o calle real.

**2. La zona peligrosa (≈ −5 a −10 dB).** Acá aparece lo que importa, en dos pasos.

A −5 dB el modelo **se rinde a mitad de camino**: transcribe bien lo que alcanza y corta la frase con puntos suspensivos o un `[inaudible]`. Es el comportamiento deseable — el propio modelo está avisando que no le da.

A −10 dB pasa otra cosa, y es la que hay que temer: **inventa**. En nuestra corrida, "Hola, soy José Manuel" salió como *"Hola, soy ChatGPT"* en las tres repeticiones. Cuando la evidencia acústica se agota, el modelo no se queda callado: cae en su **prior de lenguaje** y completa con lo que estadísticamente calza — y lo que más calza en un corpus de internet, después de "hola, soy…", no es el nombre de tu cliente. El resultado es fluido, gramatical, plausible y falso. Un `[inaudible]` se detecta y se escala a un humano; una alucinación bien escrita se cuela silenciosa hasta el reporte final.

Fíjate también en **qué** fue lo primero que se corrompió: un nombre propio. Es justo lo que el modelo de lenguaje no puede predecir del contexto, y justo lo que más suele importar en un audio de negocio — nombres de clientes, RUTs, montos, números de póliza. Y nota que las tres repeticiones alucinaron *distinto* ("Hola, ChatGPT." vs. "Hola, soy ChatGPT. Bienvenido al…"): la evidencia se acabó y cada llamada rellena a su manera.

**3. Silencio (−15 dB).** Ya no queda señal que recuperar y el modelo devuelve texto vacío de forma consistente.

> Tus resultados exactos van a diferir de los de arriba, y eso *es* el hallazgo: el audio es determinista (semilla fija) pero el modelo no. La divergencia entre repeticiones es una señal de confianza gratis que puedes usar en producción — transcribir dos veces y comparar cuesta el doble, pero te dice cuándo desconfiar.

La lección práctica: **mide la SNR de tu audio antes de culpar al modelo**, y cuando evalúes un proveedor de transcripción no mires solo su tasa de error promedio — mira qué hace en el peor caso, y si falla callado o falla inventando.

## Qué nos llevamos

- Un audio son **números**: sample rate, bit depth y canales. El sample rate pone un techo duro (Nyquist) a lo que la grabación puede contener, y ese techo el modelo no lo puede levantar.
- El **espectrograma** convierte el audio en una imagen, y esa es la razón histórica de que el reconocimiento de voz haya podido subirse a los avances de la visión por computador.
- La **voz** tiene una firma reconocible: fundamental, armónicos y formantes, con casi toda la energía útil bajo 4 kHz.
- Transcribir hoy es una llamada de API (`gpt-transcribe`), y el modelo resiste mucho más ruido del que uno supondría — pero **la calidad del audio sigue siendo el techo**. Y cuando la señal se acaba, el modelo no calla: rellena con su prior de lenguaje, y lo primero que corrompe son los nombres propios y los números. Antes de cambiar de modelo, arregla el micrófono.

En la **lección 2** recorremos el camino inverso: de texto a voz, y con un giro nuevo — hoy se le puede *dirigir* la actuación al modelo en lenguaje natural.